# Classify 3,080 real customer queries with Laya (no training)

Dataset: **Banking77** ([PolyAI](https://huggingface.co/datasets/PolyAI/banking77), CC-BY-4.0) — 13,083 real online-banking queries with 77 fine intents. We use its test split (3,080 rows) mapped to **7 coarse groups** (`data/coarse_labels.json`), because Laya's sweet spot is ≤20 options (see `TUTORIAL.md` §8).

You will learn: load data → define labels as descriptions → classify → measure accuracy/latency → read confusions → feed the Next.js demo (`web/`).

Run top to bottom. Cell 6 takes ~10 s (200 queries × ~50 ms).

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so `import laya_classify` works

import os
os.environ.setdefault("USE_TF", "0")  # always before importing laya

import csv, json, random, time
from collections import Counter

In [ ]:
LABELS = json.load(open("../data/coarse_labels.json"))  # {group: description}
rows = list(csv.DictReader(open("../data/banking77_test.csv", encoding="utf-8")))
print(f"{len(rows)} queries, {len(LABELS)} groups")
for name, desc in LABELS.items():
    print(f"  {name:<16} {desc}")

In [ ]:
def bar(n, width=40):
    return "#" * round(width * n)

counts = Counter(r["coarse_label"] for r in rows)
biggest = max(counts.values())
for name, n in counts.most_common():
    print(f"  {name:<16} {n:4d}  {bar(n / biggest)}")

In [ ]:
pick = random.Random(7).sample(rows, 5)  # fixed seed -> same peek every run
for r in pick:
    print(f"[{r['coarse_label']:<16}] {r['text']}")

## One query, one `choice` question

The 7 descriptions above **are** the classifier — no training step exists. Each option is scored at its own `[MASK]` slot in a single forward pass.

In [ ]:
from laya_classify import LayaClassifier

clf = LayaClassifier(LABELS)  # auto: MLX on Apple Silicon, else PyTorch
clf.warm()  # throwaway calls so timing starts warm

demo = "I was charged twice for the same transfer, please refund one."
pred = clf.predict(demo)
print(f"{demo}\n  -> {pred.label}  conf={pred.confidence:.3f}  ({pred.latency_ms:.1f} ms)")
for k, v in sorted(pred.probabilities.items()):
    print(f"     {k:<16} {v:.3f}")

## Classify 200 queries and score them

`test.csv` is grouped by intent, so first-200 would cover ~2 groups. Same seeded shuffle as `scripts/classify_demo_data.py` (its first 200 rows are these).

In [ ]:
N = 200
sample = rows.copy()
random.Random(7).shuffle(sample)
sample = sample[:N]

results, t0 = [], time.perf_counter()
for i, r in enumerate(sample):
    p = clf.predict(r["text"])
    results.append((r, p))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{N} ...")
print(f"done in {time.perf_counter() - t0:.1f} s")

In [ ]:
correct = sum(r["coarse_label"] == p.label for r, p in results)
print(f"accuracy: {correct}/{N} = {correct / N:.3f}  (random guessing: {1 / len(LABELS):.3f})")
ms = sum(p.latency_ms for _, p in results) / N
print(f"mean latency: {ms:.1f} ms  (~{1000 / ms:.0f} docs/s, one process)")

for name in LABELS:
    gold = [(r, p) for r, p in results if r["coarse_label"] == name]
    acc = sum(r["coarse_label"] == p.label for r, p in gold) / max(1, len(gold))
    print(f"  {name:<16} {acc:.3f}  {bar(acc)}  (n={len(gold)})")

In [ ]:
print("top confusions (gold -> predicted):")
for (g, p), n in Counter(
    (r["coarse_label"], pr.label) for r, pr in results
    if r["coarse_label"] != pr.label
).most_common(5):
    print(f"  {g} -> {p}  (x{n})")

print("\nthree mistakes up close:")
shown = 0
for r, p in results:
    if r["coarse_label"] != p.label:
        print(f"  gold={r['coarse_label']} pred={p.label} conf={p.confidence:.2f} | {r['text'][:90]}")
        shown += 1
        if shown == 3:
            break

In [ ]:
right = [p.confidence for r, p in results if r["coarse_label"] == p.label]
wrong = [p.confidence for r, p in results if r["coarse_label"] != p.label]
print(f"mean confidence when right: {sum(right) / len(right):.3f}")
print(f"mean confidence when wrong: {sum(wrong) / len(wrong):.3f}")
print("Higher when right here — but still fit temperatures on YOUR data before gating (FINDINGS.md §3).")

## Feed the web demo

This writes the same `results.json` the Next.js app renders. Then:

```bash
cd web && npm install && npm run dev   # open http://localhost:3000
```

In [ ]:
per_class = {}
for name in LABELS:
    gold = [(r, p) for r, p in results if r["coarse_label"] == name]
    ok = sum(r["coarse_label"] == p.label for r, p in gold)
    per_class[name] = {"n": len(gold), "correct": ok,
                      "accuracy": round(ok / max(1, len(gold)), 4)}

payload = {
    "meta": {"dataset": "PolyAI Banking77 test, mapped to 7 coarse groups (CC-BY-4.0)",
               "n": N, "runtime": clf.runtime,
               "accuracy": round(correct / N, 4),
               "mean_latency_ms": round(sum(p.latency_ms for _, p in results) / N, 1),
               "labels": LABELS},
    "per_class": per_class,
    "confusions": [[g, p, n] for (g, p), n in Counter(
        (r["coarse_label"], pr.label) for r, pr in results
        if r["coarse_label"] != pr.label).most_common(10)],
    "rows": [{"text": r["text"], "fine": r["fine_label"], "gold": r["coarse_label"],
               "pred": p.label, "confidence": round(p.confidence, 4),
               "correct": r["coarse_label"] == p.label,
               "probabilities": {k: round(v, 4) for k, v in p.probabilities.items()}}
              for r, p in results],
}
json.dump(payload, open("../web/public/results.json", "w"), ensure_ascii=False, indent=1)
print(f"wrote ../web/public/results.json ({N} rows, acc={correct / N:.3f})")

## Exercises

1. **Classify your own query**: `clf.predict("...")` — where does it land?
2. **Move the needle**: rewrite one description in `LABELS` and re-run cells 7–9. Descriptions are the entire training set.
3. **Add flags**: ask a `noul` (`is_urgent`) and a `score` (`frustration`) in the same `system_one` call — see `TUTORIAL.md` §5.
4. **Feel the 77-label wall**: classify into all 77 `fine_label`s in one `choice` question and watch accuracy fall (the `head_max_len` budget, `FINDINGS.md` §3).